Data Understanding → revisar ZIP → identificar hoja municipal → revisar diccionario/estructura → localizar pobreza 2020 → validar claves y faltantes → Data Preparation → construir POBREZA_2020 → auditar contra los 2,458 municipios candidatos.

FASE 2 — Data Understanding: CONEVAL 2020

# Fase 2 — Data Understanding
## Construcción y validación de CONEVAL 2020

### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa

Explorar y validar los indicadores municipales de pobreza publicados por CONEVAL.

La variable socioeconómica que se incorporará al modelo será:

`POBREZA_2020`

correspondiente al porcentaje de población en situación de pobreza en 2020.

Esta variable será utilizada posteriormente como:

`X4 = Porcentaje de población en situación de pobreza`

Antes de construirla se verificará:

- estructura interna del archivo;
- hojas disponibles;
- estructura de encabezados;
- cobertura municipal;
- claves municipales;
- disponibilidad del indicador de pobreza 2020;
- valores no disponibles;
- duplicados y valores faltantes.

En esta fase todavía no se integrará CONEVAL con las demás fuentes ni se realizarán regresiones.

In [1]:
# 2.2 Librerías y rutas
from pathlib import Path
import zipfile
from io import BytesIO

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


In [2]:
PROJECT_ROOT = Path.cwd().parent

CONEVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "coneval"
)

archivo_coneval = (
    CONEVAL_DIR
    / "Concentrado_indicadores_de_pobreza_2020.zip"
)

print("Ruta del proyecto:")
print(PROJECT_ROOT)

print("\nRuta CONEVAL:")
print(archivo_coneval)

print("\n¿Existe el archivo?:", archivo_coneval.exists())

Ruta del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta CONEVAL:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\coneval\Concentrado_indicadores_de_pobreza_2020.zip

¿Existe el archivo?: True


### 2.3 Inspección del paquete CONEVAL

Antes de cargar la información se revisa el contenido del archivo comprimido.

El objetivo es identificar el archivo Excel que contiene el concentrado de indicadores municipales de pobreza.

In [4]:
with zipfile.ZipFile(archivo_coneval, "r") as z:

    contenido_coneval = z.namelist()

    print(
        f"Número de archivos encontrados: "
        f"{len(contenido_coneval)}\n"
    )

    for nombre in contenido_coneval:
        print(nombre)

Número de archivos encontrados: 1

Concentrado_indicadores_de_pobreza_2020.xlsx


In [5]:
# 2.4 Revisar las hojas del Excel

with zipfile.ZipFile(archivo_coneval, "r") as z:

    archivo_excel_coneval = [
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".xlsx")
    ][0]

    excel_bytes = BytesIO(
        z.read(archivo_excel_coneval)
    )

excel_coneval = pd.ExcelFile(excel_bytes)

print("Hojas disponibles:")

for hoja in excel_coneval.sheet_names:
    print(hoja)

Hojas disponibles:
Concentrado municipal
Concentrado estatal


In [7]:
################2.5 Revisar primero los encabezados originales

coneval_raw = pd.read_excel(
    excel_coneval,
    sheet_name="Concentrado municipal",
    header=[4, 5]
)

print("Dimensiones iniciales:")
print(coneval_raw.shape)

print("\nNúmero de columnas iniciales:")
print(coneval_raw.shape[1])

Dimensiones iniciales:
(2487, 146)

Número de columnas iniciales:
146


In [8]:
# 2.7 Limpiar los encabezados multinivel
def limpiar_columna_coneval(columna):

    partes = []

    for elemento in columna:

        texto = str(elemento).strip()
        texto = texto.replace("\n", " ")

        if (
            texto
            and not texto.startswith("Unnamed:")
            and texto.lower() != "nan"
        ):
            partes.append(texto)

    return "__".join(partes)

In [9]:
## Limpiar columnas
coneval_raw.columns = [
    limpiar_columna_coneval(columna)
    for columna in coneval_raw.columns
]

# Eliminar columnas completamente vacías
coneval_raw = coneval_raw.loc[
    :,
    coneval_raw.columns != ""
].copy()

print("Dimensiones después de limpiar:")
print(coneval_raw.shape)

print("\nPrimeras 15 columnas:")

for i, columna in enumerate(
    coneval_raw.columns[:15],
    start=1
):
    print(f"{i:02d}. {columna}")


Dimensiones después de limpiar:
(2487, 145)

Primeras 15 columnas:
01. Clave de entidad
02. Entidad federativa
03. Clave de municipio
04. Municipio
05. Población 2010* (leer nota al final del cuadro)
06. Población 2015* (leer nota al final del cuadro)
07. Población 2020* (leer nota al final del cuadro)
08. Pobreza__Porcentaje 2010
09. Pobreza__Porcentaje 2015
10. Pobreza__Porcentaje 2020
11. Pobreza__Personas 2010
12. Pobreza__Personas 2015
13. Pobreza__Personas 2020
14. Pobreza__Carencias promedio 2010
15. Pobreza__Carencias promedio 2015


In [10]:
#2.8 Validar nuestras cinco variables

variables_coneval_requeridas = [
    "Clave de entidad",
    "Entidad federativa",
    "Clave de municipio",
    "Municipio",
    "Pobreza__Porcentaje 2020"
]

validacion_variables_coneval = pd.DataFrame({
    "variable": variables_coneval_requeridas,
    "disponible": [
        variable in coneval_raw.columns
        for variable in variables_coneval_requeridas
    ]
})

display(validacion_variables_coneval)

variables_faltantes_coneval = [
    variable
    for variable in variables_coneval_requeridas
    if variable not in coneval_raw.columns
]

print(
    "Variables faltantes:",
    variables_faltantes_coneval
)

,variable,disponible
0,Clave de entidad,True
1,Entidad federativa,True
2,Clave de municipio,True
3,Municipio,True
4,Pobreza__Porcentaje 2020,True


Variables faltantes: []


In [11]:
# 2.9 Identificar los verdaderos registros municipales

clave_municipal_numerica = pd.to_numeric(
    coneval_raw["Clave de municipio"],
    errors="coerce"
)

coneval_municipal_raw = (
    coneval_raw[
        clave_municipal_numerica.notna()
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Registros municipales identificados:",
    f"{len(coneval_municipal_raw):,}"
)

Registros municipales identificados: 2,469


In [12]:
# ########## creamos CVEGEO
coneval_municipal_raw["CVEGEO"] = (
    pd.to_numeric(
        coneval_municipal_raw["Clave de municipio"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

display(
    coneval_municipal_raw[
        [
            "CVEGEO",
            "Entidad federativa",
            "Municipio",
            "Pobreza__Porcentaje 2020"
        ]
    ].head(10)
)

,CVEGEO,Entidad federativa,Municipio,Pobreza__Porcentaje 2020
0,01001,Aguascalientes,Aguascalientes,23.682258
1,01002,Aguascalientes,Asientos,40.131881
2,01003,Aguascalientes,Calvillo,45.755944
3,01004,Aguascalientes,Cosío,37.039182
4,01005,Aguascalientes,Jesús María,26.341836
5,01006,Aguascalientes,Pabellón de Arteaga,28.91802
6,01007,Aguascalientes,Rincón de Romos,40.511419
7,01008,Aguascalientes,San José de Gracia,45.770922
8,01009,Aguascalientes,Tepezalá,37.647472
9,01010,Aguascalientes,El Llano,40.612105


In [13]:
############ 2.10 Revisar los valores no numéricos de pobreza

pobreza_original = (
    coneval_municipal_raw[
        "Pobreza__Porcentaje 2020"
    ]
    .astype("string")
    .str.strip()
)

pobreza_numerica = pd.to_numeric(
    pobreza_original,
    errors="coerce"
)

valores_no_numericos_pobreza = (
    pobreza_original[
        pobreza_numerica.isna()
        & pobreza_original.notna()
    ]
    .value_counts()
)

print(
    "Valores no numéricos encontrados "
    "en pobreza 2020:"
)

display(valores_no_numericos_pobreza)

Valores no numéricos encontrados en pobreza 2020:


Pobreza__Porcentaje 2020
n.d.    3
Name: count, dtype: Int64

In [14]:
municipios_pobreza_no_disponible = (
    coneval_municipal_raw[
        pobreza_numerica.isna()
    ][
        [
            "CVEGEO",
            "Entidad federativa",
            "Municipio",
            "Pobreza__Porcentaje 2020"
        ]
    ]
    .copy()
)

print(
    "Municipios sin indicador numérico de pobreza 2020:",
    len(municipios_pobreza_no_disponible)
)

display(municipios_pobreza_no_disponible)

Municipios sin indicador numérico de pobreza 2020: 3


,CVEGEO,Entidad federativa,Municipio,Pobreza__Porcentaje 2020
33,04012,Campeche,Seybaplaya,n.d.
205,07125,Chiapas,Honduras de la Sierra,n.d.
2080,29048,Tlaxcala,La Magdalena Tlaltelulco,n.d.


2.11 Auditoría de los n.d. contra la muestra candidata

### 2.11 Revisión de valores no disponibles frente a la muestra candidata

CONEVAL presenta tres municipios con valor `n.d.` en el porcentaje de población en situación de pobreza para 2020.

Estos valores no serán sustituidos por cero ni imputados artificialmente.

Antes de determinar su impacto sobre la muestra econométrica se verifica cuáles de estos municipios pertenecen a la muestra candidata construida previamente mediante DENUE, CONAPO, Censo e ILMM.

In [15]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

archivo_muestra_candidata = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo_ilmm.csv"
)

muestra_candidata = pd.read_csv(
    archivo_muestra_candidata,
    dtype={"CVEGEO": "string"}
)

print(
    "Municipios candidatos antes de CONEVAL:",
    f"{len(muestra_candidata):,}"
)

Municipios candidatos antes de CONEVAL: 2,458


In [16]:
claves_candidatas = set(
    muestra_candidata["CVEGEO"]
)

municipios_nd_en_muestra = (
    municipios_pobreza_no_disponible[
        municipios_pobreza_no_disponible["CVEGEO"]
        .isin(claves_candidatas)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Municipios con n.d. que pertenecen "
    "a la muestra candidata:",
    len(municipios_nd_en_muestra)
)

display(municipios_nd_en_muestra)

Municipios con n.d. que pertenecen a la muestra candidata: 1


,CVEGEO,Entidad federativa,Municipio,Pobreza__Porcentaje 2020
0,29048,Tlaxcala,La Magdalena Tlaltelulco,n.d.


In [17]:
###  2.12 Convertir pobreza a numérico

coneval_municipal_raw["POBREZA_2020"] = pd.to_numeric(
    coneval_municipal_raw[
        "Pobreza__Porcentaje 2020"
    ],
    errors="coerce"
)

In [18]:
print(
    "POBREZA_2020 nulos:",
    coneval_municipal_raw["POBREZA_2020"]
    .isna()
    .sum()
)

print(
    "POBREZA_2020 mínimo:",
    coneval_municipal_raw["POBREZA_2020"]
    .min()
)

print(
    "POBREZA_2020 máximo:",
    coneval_municipal_raw["POBREZA_2020"]
    .max()
)

POBREZA_2020 nulos: 3
POBREZA_2020 mínimo: 5.4509509601
POBREZA_2020 máximo: 99.6466761537


In [20]:
# 2.13 Validación geográfica de CONEVAL

print("CONTROL GEOGRÁFICO — CONEVAL 2020")
print("=" * 50)

print(
    "Número de municipios:",
    f"{len(coneval_municipal_raw):,}"
)

print(
    "CVEGEO únicos:",
    f"{coneval_municipal_raw['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    coneval_municipal_raw["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "POBREZA_2020 numérica disponible:",
    coneval_municipal_raw["POBREZA_2020"]
    .notna()
    .sum()
)

print(
    "POBREZA_2020 no disponible:",
    coneval_municipal_raw["POBREZA_2020"]
    .isna()
    .sum()
)

CONTROL GEOGRÁFICO — CONEVAL 2020
Número de municipios: 2,469
CVEGEO únicos: 2,469
CVEGEO duplicados: 0
POBREZA_2020 numérica disponible: 2466
POBREZA_2020 no disponible: 3


### 2.14 Cierre de Data Understanding — CONEVAL

La revisión del concentrado municipal de indicadores de pobreza permitió identificar y validar la variable socioeconómica requerida para el proyecto.

Principales resultados:

- Se utilizó la hoja `Concentrado municipal`.
- Se identificaron correctamente las claves y nombres municipales.
- Se localizó el indicador `Pobreza__Porcentaje 2020`.
- Los encabezados multinivel fueron normalizados para facilitar su procesamiento.
- Se identificaron tres municipios con valor `n.d.` en pobreza 2020.
- Los valores `n.d.` no fueron sustituidos por cero ni imputados.
- La variable de pobreza fue convertida a formato numérico, conservando los valores no disponibles como faltantes.
- Se verificó la unicidad de las claves municipales antes de construir la base procesada.

La variable socioeconómica del proyecto se define como:

`POBREZA_2020 = porcentaje de población en situación de pobreza en 2020`

y será utilizada como la variable explicativa X4 del modelo econométrico.

### FASE 3 — Data Preparation de CONEVAL

Después construiremos la base:

CVEGEO,
entidad,
municipio y
POBREZA_2020

In [21]:
coneval_2020_municipal = (
    coneval_municipal_raw[
        [
            "CVEGEO",
            "Entidad federativa",
            "Municipio",
            "POBREZA_2020"
        ]
    ]
    .copy()
    .rename(
        columns={
            "Entidad federativa": "entidad",
            "Municipio": "municipio"
        }
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Dimensiones de la base municipal CONEVAL:"
)

print(coneval_2020_municipal.shape)

display(coneval_2020_municipal.head(10))

Dimensiones de la base municipal CONEVAL:
(2469, 4)


,CVEGEO,entidad,municipio,POBREZA_2020
0,01001,Aguascalientes,Aguascalientes,23.682258
1,01002,Aguascalientes,Asientos,40.131881
2,01003,Aguascalientes,Calvillo,45.755944
3,01004,Aguascalientes,Cosío,37.039182
4,01005,Aguascalientes,Jesús María,26.341836
5,01006,Aguascalientes,Pabellón de Arteaga,28.918020
6,01007,Aguascalientes,Rincón de Romos,40.511419
7,01008,Aguascalientes,San José de Gracia,45.770922
8,01009,Aguascalientes,Tepezalá,37.647472
9,01010,Aguascalientes,El Llano,40.612105


# Fase 3 — Data Preparation
## Construcción de la base municipal CONEVAL 2020

Se construye una base con una observación por municipio y el porcentaje de población en situación de pobreza correspondiente a 2020.

Los valores `n.d.` publicados por la fuente se conservan como valores faltantes y no son imputados artificialmente.

La variable `POBREZA_2020` será utilizada como la variable explicativa socioeconómica X4 del modelo econométrico.

In [22]:
coneval_2020_municipal = (
    coneval_municipal_raw[
        [
            "CVEGEO",
            "Entidad federativa",
            "Municipio",
            "POBREZA_2020"
        ]
    ]
    .copy()
    .rename(
        columns={
            "Entidad federativa": "entidad",
            "Municipio": "municipio"
        }
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print("Dimensiones de la base municipal CONEVAL:")
print(coneval_2020_municipal.shape)

display(coneval_2020_municipal.head(10))

Dimensiones de la base municipal CONEVAL:
(2469, 4)


,CVEGEO,entidad,municipio,POBREZA_2020
0,01001,Aguascalientes,Aguascalientes,23.682258
1,01002,Aguascalientes,Asientos,40.131881
2,01003,Aguascalientes,Calvillo,45.755944
3,01004,Aguascalientes,Cosío,37.039182
4,01005,Aguascalientes,Jesús María,26.341836
5,01006,Aguascalientes,Pabellón de Arteaga,28.918020
6,01007,Aguascalientes,Rincón de Romos,40.511419
7,01008,Aguascalientes,San José de Gracia,45.770922
8,01009,Aguascalientes,Tepezalá,37.647472
9,01010,Aguascalientes,El Llano,40.612105


In [23]:
#33 3.2 Control de calidad

print("CONTROL DE CALIDAD — CONEVAL MUNICIPAL 2020")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(coneval_2020_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{coneval_2020_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    coneval_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "POBREZA_2020 disponibles:",
    coneval_2020_municipal["POBREZA_2020"]
    .notna()
    .sum()
)

print(
    "POBREZA_2020 no disponibles:",
    coneval_2020_municipal["POBREZA_2020"]
    .isna()
    .sum()
)

print(
    "POBREZA_2020 mínimo:",
    coneval_2020_municipal["POBREZA_2020"].min()
)

print(
    "POBREZA_2020 máximo:",
    coneval_2020_municipal["POBREZA_2020"].max()
)

CONTROL DE CALIDAD — CONEVAL MUNICIPAL 2020
Número de municipios: 2,469
CVEGEO únicos: 2,469
CVEGEO duplicados: 0
POBREZA_2020 disponibles: 2466
POBREZA_2020 no disponibles: 3
POBREZA_2020 mínimo: 5.4509509601
POBREZA_2020 máximo: 99.6466761537


In [24]:
# 3.3 Validaciones automáticas. Aquí no debemos exigir cero nulos, porque sabemos que los tres n.d. son legítimos.



assert (
    coneval_2020_municipal["CVEGEO"].nunique()
    == len(coneval_2020_municipal)
), "Existen problemas de unicidad."

assert (
    coneval_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    coneval_2020_municipal["POBREZA_2020"]
    .dropna()
    .between(0, 100)
    .all()
), "Existen porcentajes de pobreza fuera de rango."

assert (
    coneval_2020_municipal["POBREZA_2020"]
    .isna()
    .sum()
    == 3
), "El número de valores no disponibles cambió."

print("Todas las validaciones fueron superadas correctamente.")

Todas las validaciones fueron superadas correctamente.


In [25]:
## 3.4 Guardar la base CONEVAL
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

archivo_salida_coneval = (
    PROCESSED_DIR / "coneval_2020_municipal.csv"
)

coneval_2020_municipal.to_csv(
    archivo_salida_coneval,
    index=False,
    encoding="utf-8-sig"
)

print("Base CONEVAL 2020 guardada correctamente.")
print(archivo_salida_coneval)

Base CONEVAL 2020 guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\coneval_2020_municipal.csv


### 3.5 Auditoría de cobertura CONEVAL vs. muestra candidata

Se compara CONEVAL 2020 con la muestra candidata construida mediante DENUE, CONAPO, Censo e ILMM.

Se distingue entre:

- ausencia de correspondencia geográfica;
- presencia del municipio pero ausencia del indicador numérico de pobreza.

No se realizará imputación para valores `n.d.`.

In [26]:
claves_candidatas = set(
    muestra_candidata["CVEGEO"]
)

claves_coneval = set(
    coneval_2020_municipal["CVEGEO"]
)

claves_comunes_coneval = (
    claves_candidatas
    & claves_coneval
)

solo_candidata = (
    claves_candidatas
    - claves_coneval
)

solo_coneval = (
    claves_coneval
    - claves_candidatas
)

print(
    "Municipios candidatos:",
    f"{len(claves_candidatas):,}"
)

print(
    "Municipios CONEVAL:",
    f"{len(claves_coneval):,}"
)

print(
    "Municipios comunes:",
    f"{len(claves_comunes_coneval):,}"
)

print(
    "Candidatos sin correspondencia en CONEVAL:",
    f"{len(solo_candidata):,}"
)

print(
    "CONEVAL fuera de muestra candidata:",
    f"{len(solo_coneval):,}"
)

Municipios candidatos: 2,458
Municipios CONEVAL: 2,469
Municipios comunes: 2,458
Candidatos sin correspondencia en CONEVAL: 0
CONEVAL fuera de muestra candidata: 11


In [27]:
# 3.6 Incorporar X4 y determinar la muestra completa por disponibilidad
validacion_x4 = (
    muestra_candidata[["CVEGEO"]]
    .merge(
        coneval_2020_municipal[
            [
                "CVEGEO",
                "POBREZA_2020"
            ]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Municipios candidatos:",
    f"{len(validacion_x4):,}"
)

print(
    "POBREZA_2020 disponible:",
    f"{validacion_x4['POBREZA_2020'].notna().sum():,}"
)

print(
    "POBREZA_2020 faltante:",
    f"{validacion_x4['POBREZA_2020'].isna().sum():,}"
)


Municipios candidatos: 2,458
POBREZA_2020 disponible: 2,457
POBREZA_2020 faltante: 1


In [28]:
# 3.7 Identificar la observación que se pierde

detalle_x4_faltante = validacion_x4[
    validacion_x4["POBREZA_2020"].isna()
].copy()

display(detalle_x4_faltante)

,CVEGEO,POBREZA_2020
2069,29048,NaN


In [29]:
# 3.8 Construir la muestra candidata con cobertura completa

muestra_candidata_cobertura_completa = (
    validacion_x4[
        validacion_x4["POBREZA_2020"]
        .notna()
    ][["CVEGEO"]]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios con cobertura completa Y, X1, X2, X3 y X4:",
    f"{len(muestra_candidata_cobertura_completa):,}"
)

print(
    "CVEGEO únicos:",
    muestra_candidata_cobertura_completa[
        "CVEGEO"
    ].nunique()
)

Municipios con cobertura completa Y, X1, X2, X3 y X4: 2,457
CVEGEO únicos: 2457


### 3.9 Exportación de la muestra candidata con cobertura completa

Después de incorporar la información de CONEVAL y excluir los municipios sin disponibilidad numérica de `POBREZA_2020`, se guarda la muestra candidata con cobertura completa.

La muestra contiene 2,457 municipios con disponibilidad de información para los componentes de la variable dependiente y las cuatro variables explicativas del modelo.

Esta muestra todavía no se considera definitiva, ya que posteriormente será sometida a la auditoría de comparabilidad territorial entre 2020 y 2025.

In [30]:
archivo_muestra_cobertura_completa = (
    PROCESSED_DIR
    / "muestra_candidata_cobertura_completa.csv"
)

muestra_candidata_cobertura_completa.to_csv(
    archivo_muestra_cobertura_completa,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Muestra candidata con cobertura completa "
    "guardada correctamente."
)

print(f"Ruta: {archivo_muestra_cobertura_completa}")

print(
    "¿Existe el archivo?:",
    archivo_muestra_cobertura_completa.exists()
)

Muestra candidata con cobertura completa guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\muestra_candidata_cobertura_completa.csv
¿Existe el archivo?: True


In [31]:
####  3.10 Verificación del archivo guardado

verificacion_muestra = pd.read_csv(
    archivo_muestra_cobertura_completa,
    dtype={"CVEGEO": "string"}
)

print(
    "Dimensiones:",
    verificacion_muestra.shape
)

print(
    "CVEGEO únicos:",
    verificacion_muestra["CVEGEO"].nunique()
)

print(
    "CVEGEO duplicados:",
    verificacion_muestra["CVEGEO"]
    .duplicated()
    .sum()
)

display(verificacion_muestra.head(10))

Dimensiones: (2457, 1)
CVEGEO únicos: 2457
CVEGEO duplicados: 0


,CVEGEO
0,01001
1,01002
2,01003
3,01004
4,01005
5,01006
6,01007
7,01008
8,01009
9,01010


### Conclusión de la preparación de CONEVAL 2020

La base municipal CONEVAL 2020 quedó construida y validada correctamente.

Principales resultados:

- CONEVAL contiene 2,469 municipios.
- Las 2,469 claves municipales son únicas.
- No existen claves duplicadas.
- Se identificaron 2,466 municipios con un valor numérico de pobreza 2020.
- Tres municipios presentan el valor `n.d.`.
- Los valores `n.d.` no fueron imputados artificialmente.
- De los tres municipios con `n.d.`, únicamente La Magdalena Tlaltelulco (`29048`) pertenece a la muestra candidata previa.
- La incorporación de CONEVAL reduce la muestra candidata de 2,458 a 2,457 municipios con información completa.
- Los municipios restantes cuentan con `POBREZA_2020`.

Por tanto, `POBREZA_2020` queda disponible como la variable explicativa socioeconómica X4 del modelo econométrico.

La muestra de 2,457 municipios representa la cobertura completa de las variables del modelo, pero todavía deberá someterse a la revisión de comparabilidad territorial 2020–2025 antes de considerarse la muestra econométrica definitiva.